# TP 2 — Premiers pas PySpark

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Ce notebook guide les parties B (WordCount), C (fil rouge) et D (Spark UI)
du TP 2. Complétez les cellules marquées `À COMPLÉTER`, exécutez tout de bout
en bout, puis poussez le notebook **avec ses sorties** dans `notebooks/`.

**Livrable** : ce notebook, avec le WordCount **commenté ligne à ligne**, les
observations de la Spark UI, et la comparaison Spark vs Pandas.

> Réflexe d'ingénieur : on ne dit pas « c'est lent », on dit « 4,2 s pour
> 50 000 lignes ». Mesurez tout.

## 0. Vérification de l'environnement

PySpark s'exécute sur la JVM : un **JDK 17** est requis. Si la cellule
échoue, revenez à la partie A du TP (`java -version`).

In [7]:
import sys, platform
import pyspark
print("Python  :", sys.version.split()[0], "-", platform.system())
print("PySpark :", pyspark.__version__)

Python  : 3.12.2 - Windows
PySpark : 3.5.1


## 1. Créer la SparkSession

`master("local[*]")` exécute Spark dans ce notebook, sur **tous les cœurs**
de la machine. La Spark UI démarre sur http://localhost:4040 — ouvrez-la.

In [8]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("TP2-WordCount")
         .master("local[*]")
         .getOrCreate())

print("Spark", spark.version)
print("Coeurs vus :", spark.sparkContext.defaultParallelism)
print("Spark UI   :", spark.sparkContext.uiWebUrl)

Spark 3.5.1
Coeurs vus : 8
Spark UI   : http://DESKTOP-5NL6LPD.home:4040


## Exercice B — WordCount

Préparez d'abord un fichier texte dans `data/discours.txt` (un discours, un
article — au moins quelques centaines de mots). La cellule ci-dessous compte
les mots. **Commentez chaque ligne** : c'est le cœur du livrable.

In [10]:
from pyspark.sql.functions import explode, split, lower, col

# Lecture du fichier texte : chaque ligne du fichier devient une ligne du DataFrame,
# dans une colonne unique nommée "value". Transformation paresseuse (rien n'est encore lu).
texte = spark.read.text("../data/discours.txt")

mots = (texte
        # lower() met tout en minuscules, split(..., r"\s+") découpe chaque ligne
        # en liste de mots sur les espaces/tabulations/retours à la ligne.
        # explode() transforme cette liste en une ligne par mot (1 ligne texte -> N lignes mots).
        # Toutes ces opérations sont des transformations paresseuses.
        .select(explode(split(lower(col("value")), r"\s+")).alias("mot"))
        # filter() élimine les chaînes vides éventuelles (ex: doubles espaces). Transformation.
        .filter(col("mot") != "")
        # groupBy("mot") regroupe toutes les occurrences d'un même mot ensemble.
        # Cette opération nécessite un SHUFFLE : les données doivent être redistribuées
        # entre les partitions pour que toutes les occurrences d'un mot donné
        # se retrouvent sur le même nœud/tâche avant l'agrégation.
        .groupBy("mot")
        # count() compte les occurrences par groupe. Toujours une transformation
        # (le calcul n'est pas encore déclenché).
        .count())

# show(15) est une ACTION : c'est elle qui déclenche réellement l'exécution
# de tout le plan décrit ci-dessus (lecture, split, explode, filter, groupBy, count).
mots.orderBy(col("count").desc()).show(15)

+----------+-----+
|       mot|count|
+----------+-----+
|        de|   16|
|        la|   16|
|       des|   13|
|       les|   12|
|        le|   11|
|        et|    9|
|   données|    7|
|       ces|    4|
|        en|    4|
|traitement|    4|
|        du|    3|
|       sur|    3|
|      pour|    3|
|         à|    3|
|      data|    3|
+----------+-----+
only showing top 15 rows



### B — Observer (répondez en markdown)

1. Quelles lignes sont des **transformations** ? Laquelle est l'**action** ?
2. Où se situe le **shuffle** ?
3. Si vous relancez `mots.orderBy(...).show()`, pourquoi tout est-il
   recalculé ?

*Vos réponses :* …

 **Transformations (paresseuses)** : `spark.read.text(...)`, `select(explode(split(...)))`, `filter(...)`, `groupBy(...)`, `count()` ne font que décrire le plan d'exécution, sans rien calculer. **L'action** est `show(15)` : c'est elle qui déclenche réellement l'exécution de tout le pipeline.

2. Le **shuffle** se situe au niveau de `groupBy("mot")` : Spark doit redistribuer les données entre les partitions pour que toutes les occurrences d'un même mot se retrouvent ensemble avant d'être comptées.

3. Spark ne conserve pas le résultat calculé en mémoire par défaut (pas de `cache()`/`persist()`). Chaque action relance donc **tout le plan depuis le départ** (lecture du fichier, split, explode, filter, groupBy, count) : c'est le principe de l'évaluation paresseuse — rien n'est stocké entre deux actions sauf demande explicite.

## Exercice C — Recharger le fil rouge avec Spark

On reprend les fichiers du fil rouge, cette fois avec Spark. `inferSchema`
demande à Spark de **deviner** les types (pratique mais coûteux : une passe
de lecture en plus).

In [13]:

# Lecture du CSV : header=True utilise la première ligne comme noms de colonnes,
# inferSchema=True demande à Spark de deviner les types (coûteux : relit tout le fichier).
orders = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("../data/orders.csv"))

# read.json lit nativement le JSON Lines (un objet JSON par ligne) et infère la structure.
events = spark.read.json("../data/events.json")   # JSON Lines

orders.printSchema()
print("orders :", orders.count(), "lignes")   # count() est une action
events.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

orders : 50000 lignes
root
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ville: string (nullable = true)



### C — Explorer sans tout rapatrier

`show(n)` et `count()` sont des actions ; `select`, `filter`, `groupBy`
décrivent seulement le plan. **Ne jamais** faire `events.collect()`.

In [ ]:
# Aperçu de quelques colonnes (n lignes vers le driver, raisonnable)
orders.select("order_id", "statut", "canal").show(5)

# Filtrer puis compter (filter = transformation, count() = action)
livrees = orders.filter(col("statut") == "livrée")
print("livrees :", livrees.count())

# Compter par groupe
orders.groupBy("statut").count().show()

# Compter par canal, trié par count décroissant
orders.groupBy("canal").count().orderBy(col("count").desc()).show()

+--------+-------+----------+
|order_id| statut|     canal|
+--------+-------+----------+
|O0000001| livrée|mobile_app|
|O0000002| livrée|mobile_app|
|O0000003| livrée|       web|
|O0000004| livrée|mobile_app|
|O0000005|annulée|       web|
+--------+-------+----------+
only showing top 5 rows

livrees : 0
+---------+-----+
|   statut|count|
+---------+-----+
|retournée| 2484|
|   livrée|38890|
| en_cours| 4058|
|  annulée| 4568|
+---------+-----+

+----------+-----+
|     canal|count|
+----------+-----+
|mobile_app|32519|
|       web|17481|
+----------+-----+



## Exercice C.3 — Spark vs Pandas : mesurez

Chronométrez Spark sur `orders.csv`, comparez à votre mesure Pandas du TP 1
(même fichier, échelle 0.1). Lequel gagne sur ce petit volume ?

In [ ]:
# Lecture du CSV : header=True utilise la première ligne comme noms de colonnes,
# inferSchema=True demande à Spark de deviner les types (coûteux : relit tout le fichier).
orders = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("../data/orders.csv"))

# read.json lit nativement le JSON Lines (un objet JSON par ligne) et infère la structure.
events = spark.read.json("../data/events.json")   # JSON Lines

orders.printSchema()
print("orders :", orders.count(), "lignes")   # count() est une action
events.printSchema()

## Exercice D — Lire la Spark UI

Ouvrez http://localhost:4040. Cette cellule donne les partitions ; le reste
s'observe **dans le navigateur** (onglets Jobs, Stages, SQL/DataFrame).

In [ ]:
# === À COMPLÉTER ===
print("Partitions orders :", orders.rdd.getNumPartitions())
print("Coeurs disponibles :", spark.sparkContext.defaultParallelism)

# Relancez le WordCount pour le retrouver dans l'onglet Jobs :
mots.orderBy(col("count").desc()).show(5)

NameError: name 'orders' is not defined

### D — Relevés (complétez en markdown)

- Nombre de stages du WordCount : …
- Shuffle Read / Write du stage d'agrégation : …
- Nombre de tasks par stage / nombre de partitions : …
- Tasks en parallèle vs nombre de cœurs : …

## 5. Avant de pousser

Vérifiez : WordCount commenté ligne à ligne, observations Spark UI
renseignées, comparaison Spark/Pandas chiffrée. Puis :

```bash
git add notebooks/TP2_wordcount.ipynb
git commit -m "TP2 : WordCount PySpark, fil rouge, Spark UI"
git push
```

Pensez à **arrêter la session** en fin de travail : `spark.stop()`.

In [ ]:
spark.stop()